# REV10: baseline pembanding dan ablasi normalisasi spasi operator (SQLi, Kaggle sajid576)

Notebook ini **tidak mengubah** eksperimen REV9. Notebook ini menambah dua pengujian dengan data, split 80:20 (seed 42), dan TF-IDF yang sama:

1. **Baseline (butir 5):** Logistic Regression, Multinomial Naive Bayes, libinjection, dan OWASP CRS v4.30.0 (rule file 942, PL1 dan PL2, anomaly scoring ambang 5).
2. **Ablasi normalisasi operator (butir 6):** `normalize_sql_operators()` diterapkan pada data latih, test set Kaggle, dan payload sqlmap; semua model dilatih ulang dengan hyperparameter REV9.

Hyperparameter model utama dan angka REV9 untuk cek reproduksi ditulis dari hasil Colab REV9 Anda (sel 3). Jalankan semua sel (±20–30 menit, runtime CPU). Sel terakhir mengunduh `results_rev10.zip`.

In [ ]:
import kagglehub, os, glob

# Download latest version
path = kagglehub.dataset_download("sajid576/sql-injection-dataset")
print("Path to dataset files:", path)

files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
print("CSV files:", files)
DATA_PATH = next(f for f in files if os.path.basename(f) == "Modified_SQL_Dataset.csv")
print("Using:", DATA_PATH)

In [ ]:
# Hasil REV9 (Colab Anda, 21 Nov 2026): hyperparameter 80:20 dan angka untuk cek reproduksi
import os; os.makedirs('/content/ref_rev9', exist_ok=True)

In [ ]:
%%writefile /content/ref_rev9/best_hyperparameters.csv
Split,Algorithm,cv_macro_f1,params,seconds
80:20,KNN,0.9781431083836468,"{""metric"": ""cosine"", ""n_neighbors"": 5, ""weights"": ""distance""}",133.20852828025818
80:20,SVM,0.9969816195858566,"{""C"": 10, ""gamma"": ""scale"", ""kernel"": ""linear""}",206.43142700195312
80:20,Random Forest,0.9951425870117269,"{""n_estimators"": 100, ""min_samples_split"": 2, ""min_samples_leaf"": 1, ""max_depth"": null, ""class_weight"": null}",115.09146690368652
80:20,XGBoost,0.9954691194013844,"{""subsample"": 1.0, ""n_estimators"": 200, ""max_depth"": 4, ""learning_rate"": 0.1, ""colsample_bytree"": 0.8}",250.65494585037231
70:30,KNN,0.979021964173448,"{""metric"": ""cosine"", ""n_neighbors"": 7, ""weights"": ""distance""}",122.3701651096344
70:30,SVM,0.9964419714947246,"{""C"": 10, ""gamma"": ""scale"", ""kernel"": ""linear""}",210.55904293060303
70:30,Random Forest,0.9949267508871267,"{""n_estimators"": 100, ""min_samples_split"": 2, ""min_samples_leaf"": 1, ""max_depth"": null, ""class_weight"": null}",120.22317099571228
70:30,XGBoost,0.9959023320626629,"{""subsample"": 1.0, ""n_estimators"": 200, ""max_depth"": 4, ""learning_rate"": 0.1, ""colsample_bytree"": 0.8}",257.9277904033661
90:10,KNN,0.97825548945284,"{""metric"": ""cosine"", ""n_neighbors"": 5, ""weights"": ""distance""}",124.42880773544312
90:10,SVM,0.9971974261442825,"{""C"": 10, ""gamma"": ""scale"", ""kernel"": ""linear""}",203.4554352760315
90:10,Random Forest,0.9960083568947935,"{""n_estimators"": 300, ""min_samples_split"": 2, ""min_samples_leaf"": 1, ""max_depth"": 100, ""class_weight"": ""balanced""}",111.13938736915588
90:10,XGBoost,0.9964436407677087,"{""subsample"": 1.0, ""n_estimators"": 200, ""max_depth"": 6, ""learning_rate"": 0.3, ""colsample_bytree"": 0.8}",243.6334629058838


In [ ]:
%%writefile /content/ref_rev9/results_all_splits.csv
Algorithm,Split,Accuracy,Precision,Recall,F1,F1_macro,ROC_AUC,TN,FP,FN,TP,FPR,FNR,train_s,predict_s,predict_ms_per_payload
KNN,80:20,0.9896120759616945,0.9897413024085637,0.981858407079646,0.9857840959573523,0.9887999763674229,0.9978706957371948,3878,23,41,2219,0.005895924122019995,0.018141592920353982,1.6754956245422363,11.675286293029785,1.895031049022851
SVM,80:20,0.996916085051128,0.9982214317474433,0.9933628318584071,0.9957862053670437,0.9966771805025414,0.9992044245519075,3897,4,15,2245,0.0010253781081773904,0.00663716814159292,23.014943599700928,5.3621509075164795,0.8703377548314364
Random Forest,80:20,0.9967537737380295,0.9995539696699376,0.9915929203539823,0.9955575299866726,0.9964999926148197,0.9995916068718482,3900,1,19,2241,0.0002563445270443476,0.008407079646017699,10.45912218093872,0.6806752681732178,0.11048129657088424
XGBoost,80:20,0.996266839798734,0.9991075412762159,0.9907079646017699,0.9948900244390135,0.9959746120149293,0.9996701549182987,3899,2,21,2239,0.0005126890540886952,0.009292035398230088,37.74763202667236,0.5361168384552002,0.08701782802389225
KNN,70:30,0.9898279406990586,0.9869976359338062,0.9852507374631269,0.9861234130498967,0.9890473573619829,0.9975149345421125,5807,44,50,3340,0.0075200820372585886,0.014749262536873156,1.5101690292358398,15.462111711502075,1.6732076302891543
SVM,70:30,0.9971864516827184,0.9988137603795967,0.9935103244837759,0.9961549837326235,0.9969682768492469,0.9990240429868782,5847,4,22,3368,0.0006836438215689625,0.006489675516224189,21.032011032104492,7.637879848480225,0.8265209228958148
Random Forest,70:30,0.9971864516827184,1.0,0.9923303834808259,0.996150429375185,0.9969667562974152,0.9994838640395787,5851,0,26,3364,0.0,0.007669616519174041,9.275376558303833,0.9608101844787598,0.10397253376028133
XGBoost,70:30,0.9966453846986257,0.9994052928932501,0.9914454277286135,0.995409447652895,0.9963832130412142,0.9997195346180392,5849,2,29,3361,0.0003418219107844813,0.00855457227138643,34.67815327644348,0.7737417221069336,0.08372922000940738
KNN,90:10,0.991236611489776,0.9902222222222222,0.9858407079646018,0.9880266075388027,0.9905579671940238,0.9970396393045545,1940,11,16,1114,0.005638134290107637,0.01415929203539823,1.8812904357910156,5.8664562702178955,1.9040753879318064
SVM,90:10,0.997078870496592,0.9991095280498664,0.9929203539823008,0.996005326231691,0.9968514735532976,0.9988728267328304,1950,1,8,1122,0.0005125576627370579,0.007079646017699115,25.43544840812683,2.622074604034424,0.8510466095535293
Random Forest,90:10,0.9954560207724765,1.0,0.9876106194690265,0.993766696349065,0.9950958098701403,0.9999369508715749,1951,0,14,1116,0.0,0.012389380530973451,31.687674522399902,0.9136874675750732,0.29655549093640804
XGBoost,90:10,0.9974034404414152,1.0,0.9929203539823008,0.9964476021314387,0.9972007831629061,0.9998784376516695,1951,0,8,1122,0.0,0.007079646017699115,52.60480356216431,0.26442646980285645,0.08582488471368271


In [ ]:
%%writefile /content/ref_rev9/external_sqlmap_summary.json
{
  "generated": 16440,
  "unique": 15353,
  "in_dataset": 21,
  "evaluated": 15332,
  "level1": 181,
  "detection_all": {
    "KNN": 96.33446386642316,
    "SVM": 96.16488390294808,
    "Random Forest": 94.2081920166971,
    "XGBoost": 96.02791547091051
  },
  "detection_spaced": {
    "KNN": 99.23689016436212,
    "SVM": 99.7978085050874,
    "Random Forest": 99.73910774849986,
    "XGBoost": 98.68901643621184
  },
  "share_spaced_equals": {
    "orig": 0.0,
    "spaced": 64.31646230106965
  },
  "detection_level1": {
    "KNN": 98.34254143646409,
    "SVM": 97.79005524861878,
    "Random Forest": 90.60773480662984,
    "XGBoost": 90.60773480662984
  },
  "by_technique": {
    "UNION query": {
      "KNN": 99.9804992199688,
      "SVM": 100.0,
      "Random Forest": 100.0,
      "XGBoost": 100.0,
      "n": 5128
    },
    "boolean-based blind": {
      "KNN": 85.44061302681992,
      "SVM": 98.02043422733078,
      "Random Forest": 98.4035759897829,
      "XGBoost": 94.44444444444444,
      "n": 1566
    },
    "error-based": {
      "KNN": 96.97766097240473,
      "SVM": 96.58344283837057,
      "Random Forest": 97.43758212877792,
      "XGBoost": 99.24441524310119,
      "n": 3044
    },
    "inline query": {
      "KNN": 100.0,
      "SVM": 100.0,
      "Random Forest": 87.5,
      "XGBoost": 100.0,
      "n": 8
    },
    "stacked queries": {
      "KNN": 97.95221843003414,
      "SVM": 93.31058020477816,
      "Random Forest": 79.11262798634813,
      "XGBoost": 87.30375426621161,
      "n": 1465
    },
    "time-based blind": {
      "KNN": 94.87988352341664,
      "SVM": 91.38558602280999,
      "Random Forest": 88.40087357437515,
      "XGBoost": 92.4047561271536,
      "n": 4121
    }
  },
  "kaggle_sqli_test_spaced_equals": 80.53097345132744
}


In [ ]:
%%writefile /content/gen_sqlmap.py
"""Assemble concrete SQLi strings from sqlmap's own test templates (data/xml/payloads/*.xml)
and boundaries (data/xml/boundaries.xml), following lib/controller/checks.py and lib/core/agent.py
(prefixQuery/suffixQuery). Strings are only generated for offline classification; nothing is sent."""
import xml.etree.ElementTree as ET, glob, random, re, sys, csv, os
X = sys.argv[1]; OUT = sys.argv[2]
rng = random.Random(42)
def rint(n=4): return str(rng.randint(10**(n-1), 10**n-1))
def rstr(n=4): return ''.join(rng.choice('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ') for _ in range(n))
def rng_list(s):
    out=set()
    for part in (s or '').split(','):
        part=part.strip()
        if not part: continue
        if '-' in part: a,b=part.split('-'); out |= set(range(int(a),int(b)+1))
        else: out.add(int(part))
    return out
def fill(s):
    if s is None: return ''
    s = s.replace('[GENERIC_SQL_COMMENT]', '-- [RANDSTR]')
    for k in ['[RANDNUM5]','[RANDNUM4]','[RANDNUM3]','[RANDNUM2]','[RANDNUM1]','[RANDNUM]']:
        while k in s: s = s.replace(k, rint(), 1)
    while '[RANDSTR]' in s: s = s.replace('[RANDSTR]', rstr(), 1)
    s = s.replace('[SLEEPTIME]', '5').replace('[ORIGVALUE]', '1')
    s = s.replace('[DELIMITER_START]', 'q'+''.join(rng.choice('zxjkvbp') for _ in range(3))+'q')
    s = s.replace('[DELIMITER_STOP]', 'q'+''.join(rng.choice('zxjkvbp') for _ in range(3))+'q')
    return s
B=[]
for b in ET.parse(os.path.join(X,'boundaries.xml')).getroot().findall('boundary'):
    B.append(dict(level=int(b.findtext('level')), clause=rng_list(b.findtext('clause')), where=rng_list(b.findtext('where')),
                  prefix=b.findtext('prefix') or '', suffix=b.findtext('suffix') or ''))
rows=[]
STYPE={1:'boolean-based blind',2:'error-based',3:'inline query',4:'stacked queries',5:'time-based blind',6:'UNION query'}
for f in sorted(glob.glob(os.path.join(X,'payloads','*.xml'))):
    for t in ET.parse(f).getroot().findall('test'):
        st=int(t.findtext('stype')); lvl=int(t.findtext('level')); risk=int(t.findtext('risk'))
        tcl=rng_list(t.findtext('clause')); twh=rng_list(t.findtext('where'))
        req=t.find('request'); pay=req.findtext('payload'); com=req.findtext('comment')
        if st==6:
            ch=req.findtext('char'); bodies=[]
            for n in (1,3,5,10):
                c=ch if ch and ch!='[CHAR]' else '[RANDNUM]'
                bodies.append('UNION ALL SELECT '+','.join([c]*n))
        else:
            bodies=[pay]
        for body in bodies:
            for b in B:
                if not (0 in tcl or 0 in b['clause'] or tcl & b['clause']): continue
                for w in sorted(twh):
                    if w in (1,2) and w not in b['where']: continue
                    orig = '1' if w==1 else ('-'+rint() if w==2 else '')
                    if w==3: q=''
                    elif st==4: q=b['prefix']
                    else:
                        q=b['prefix']
                        if not (body and body[0]==';') and not (q and q[-1] in '()' and body and body[0] in '()') and not (q and q[-1]=='('): q+=' '
                    expr=q+body
                    if com: expr+=com
                    elif w!=3 and b['suffix']:
                        if re.search(r'\w\Z',expr) and re.search(r'\A\w',b['suffix']): expr+=' '
                        expr+=b['suffix']
                    expr=re.sub(r';\W*;',';',expr)
                    rows.append(dict(technique=STYPE[st], level=max(lvl,b['level']), risk=risk, title=t.findtext('title'), payload=fill(orig+expr)))
                if 3 in twh: break  # REPLACE ignores boundaries: one instance is enough
with open(OUT,'w',newline='') as fh:
    wr=csv.DictWriter(fh,fieldnames=list(rows[0])); wr.writeheader(); wr.writerows(rows)
print(len(rows))


In [ ]:
!pip -q install sqlmap==1.10.11 libinjection-python==1.1.6 regex tabulate
import sqlmap, subprocess, sys, os
XMLDIR = os.path.join(os.path.dirname(sqlmap.__file__), 'data', 'xml')
subprocess.run([sys.executable, '/content/gen_sqlmap.py', XMLDIR, '/content/sqlmap_payloads.csv'], check=True)

In [ ]:
%%writefile /content/sqli_baselines_norm.py
# -*- coding: utf-8 -*-
"""
Tambahan eksperimen REV10 untuk naskah JISI (deteksi SQLi, dataset Kaggle sajid576).

  Butir 5  Baseline pembanding
           - Logistic Regression dan Multinomial Naive Bayes (TF-IDF char 1-3, split 80:20 yang sama)
           - libinjection (libinjection-python) dan OWASP CRS v4.30.0 rule file 942 (SQLi),
             dievaluasi dengan anomaly scoring CRS (CRITICAL = 5, WARNING = 3, ambang 5) pada PL1 dan PL2
  Butir 6  Ablasi normalisasi spasi operator: normalize_sql_operators() diterapkan pada data latih,
           test set Kaggle, dan payload sqlmap; semua model dilatih ulang dengan hyperparameter yang sama.

Protokol mengikuti sqli_pipeline.py (REV9): normalisasi dasar (URL decode, HTML unescape, huruf kecil,
rapatkan spasi), buang teks kosong & berlabel ganda, deduplikasi sebelum split, split bertingkat 80:20
seed 42, TF-IDF char 1-3 (20.000 fitur, sublinear, dilatih hanya pada data latih).

Pemakaian (Colab):
  !pip -q install libinjection-python==1.1.6 regex tabulate
  !python sqli_baselines_norm.py --data "$DATA_PATH" --sqlmap sqlmap_payloads.csv \
        --hp results_sqli/best_hyperparameters.csv --ref results_sqli --out results_rev10
Keluaran: CSV per tabel, tables_rev10.md (Tabel A dan B), run_info_rev10.json.
"""
import os, re, sys, io, json, time, html, argparse, hashlib, platform, urllib.request
import numpy as np, pandas as pd
from urllib.parse import unquote_plus
from scipy.stats import binomtest
import sklearn, xgboost
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from xgboost import XGBClassifier

RS, TUNE_SUB, TEST_SIZE = 42, 10000, 0.20
CRS_TAG = 'v4.30.0'
CRS_URL = f'https://raw.githubusercontent.com/coreruleset/coreruleset/{CRS_TAG}/rules/REQUEST-942-APPLICATION-ATTACK-SQLI.conf'
def log(*a): print(*a, flush=True)

# =====================================================================================
# 1. Normalisasi
# =====================================================================================
def normalize(s):
    """Normalisasi dasar REV9 (dipakai semua kondisi)."""
    s = str(s)
    try: s = unquote_plus(s)
    except Exception: pass
    s = html.unescape(s).lower()
    return re.sub(r'\s+', ' ', s).strip()

# Token multi-karakter didahulukan agar '--' tidak terpecah menjadi '-' '-', '/*' menjadi '/' '*', dst.
_OPS = r"!=|<>|<=|>=|--|/\*|\*/|[=<>(),'\"+\-*/]"
_OP_RE = re.compile(r'\s*(' + _OPS + r')\s*')
def normalize_sql_operators(text):
    """Setiap operator SQL / karakter khusus diapit tepat satu spasi, lalu spasi dirapatkan.
    Contoh: "1 and 6820=4432--" -> "1 and 6820 = 4432 --";  "1' or '1' = '1" -> "1 ' or ' 1 ' = ' 1"."""
    t = _OP_RE.sub(r' \1 ', str(text))
    return re.sub(r'\s+', ' ', t).strip()

# =====================================================================================
# 2. Data
# =====================================================================================
def load_kaggle(path):
    head = open(path, 'rb').read(4)
    enc = 'utf-16' if head[:2] in (b'\xff\xfe', b'\xfe\xff') else 'utf-8-sig'
    raw = pd.read_csv(path, encoding=enc)
    col = [c for c in raw.columns if c.lower() in ('query', 'sentence')][0]
    raw = raw.rename(columns={col: 'text', 'Label': 'label'})[['text', 'label']]
    df = raw.dropna(subset=['text']).copy(); df['text'] = df.text.astype(str)
    df['norm'] = df.text.map(normalize); df = df[df.norm != '']
    df['label'] = df.label.astype(int)
    k = df.groupby('norm').label.nunique(); conflict = set(k[k > 1].index)
    df = df[~df.norm.isin(conflict)].drop_duplicates('norm').reset_index(drop=True)
    meta = dict(file=os.path.basename(path), sha256=hashlib.sha256(open(path, 'rb').read()).hexdigest(),
                rows_raw=len(raw), rows_clean=len(df))
    return df, meta

def load_sqlmap(path, kaggle_norm):
    S = pd.read_csv(path); S['norm'] = S.payload.map(normalize)
    n_gen = len(S); S = S.drop_duplicates('norm'); n_uni = len(S)
    S = S[~S.norm.isin(set(kaggle_norm))].reset_index(drop=True)      # sama dengan REV9
    return S, dict(generated=n_gen, unique=n_uni, evaluated=len(S))

# =====================================================================================
# 3. Model ML
# =====================================================================================
def vectorizer():
    return TfidfVectorizer(analyzer='char', ngram_range=(1, 3), max_features=20000, sublinear_tf=True,
                           lowercase=False, dtype=np.float32)

MODELS = {   # est: konstruktor; space: ruang tuning (hanya dipakai bila hyperparameter belum ada)
 'KNN': dict(est=lambda: KNeighborsClassifier(algorithm='brute', n_jobs=-1),
             space={'clf__n_neighbors': [3, 5, 7, 9], 'clf__weights': ['uniform', 'distance'], 'clf__metric': ['cosine', 'euclidean']}),
 'SVM': dict(est=lambda: SVC(cache_size=1000, random_state=RS),
             space={'clf__kernel': ['linear', 'rbf'], 'clf__C': [0.1, 1, 10], 'clf__gamma': ['scale']}),
 'Random Forest': dict(est=lambda: RandomForestClassifier(random_state=RS, n_jobs=-1), space=None),
 'XGBoost': dict(est=lambda: XGBClassifier(objective='binary:logistic', eval_metric='logloss', tree_method='hist',
                                           random_state=RS, n_jobs=-1, verbosity=0), space=None, sw=True),
 # ---- baseline linear (Butir 5) ----
 'Logistic Regression': dict(est=lambda: LogisticRegression(solver='liblinear', max_iter=2000, random_state=RS),
             space={'clf__C': [0.1, 1, 10, 100], 'clf__class_weight': [None, 'balanced']}),
 'Naive Bayes': dict(est=lambda: MultinomialNB(),
             space={'clf__alpha': [0.01, 0.1, 0.5, 1.0]}),
}
MAIN = ['KNN', 'SVM', 'Random Forest', 'XGBoost']; LINEAR = ['Logistic Regression', 'Naive Bayes']

def make_pipe(name, params):
    p = Pipeline([('vec', vectorizer()), ('clf', MODELS[name]['est']())]); p.set_params(**params); return p
def fit_kw(name, y): return {'clf__sample_weight': compute_sample_weight('balanced', y)} if MODELS[name].get('sw') else {}

def tune(name, X, y, tr):
    """Protokol tuning REV9: subsampel bertingkat 10.000 data latih, CV 3-fold, macro-F1."""
    sub, _ = train_test_split(tr, train_size=TUNE_SUB, stratify=y[tr], random_state=RS)
    s = GridSearchCV(make_pipe(name, {}), MODELS[name]['space'], scoring='f1_macro',
                     cv=StratifiedKFold(3, shuffle=True, random_state=RS), n_jobs=1, refit=False)
    s.fit(X[sub], y[sub]); return s.best_params_, float(s.best_score_)

def load_hp(path):
    """best_hyperparameters.csv dari run REV9 (Colab); hanya baris split 80:20."""
    hp = {}
    if path and os.path.exists(path):
        h = pd.read_csv(path); h = h[h.Split == '80:20']
        for r in h.itertuples(): hp[r.Algorithm] = {'clf__' + k: v for k, v in json.loads(r.params).items()}
    return hp

# =====================================================================================
# 4. Baseline rule-based
# =====================================================================================
def libinjection_fn():
    try:
        import libinjection
        return lambda s: bool(libinjection.is_sql_injection(s)['is_sqli']), 'libinjection-python ' + _pkgver('libinjection-python')
    except Exception as e:
        log('[PERINGATAN] libinjection tidak tersedia:', e); return None, None

def _pkgver(p):
    try:
        from importlib.metadata import version; return version(p)
    except Exception: return '?'

def _crs_transform(s, tfs):
    for t in tfs:
        if t == 'urlDecodeUni':
            try: s = unquote_plus(s)
            except Exception: pass
        elif t == 'replaceComments': s = re.sub(r'/\*.*?(\*/|$)', ' ', s, flags=re.S)
        elif t == 'removeCommentsChar': s = re.sub(r'/\*|\*/|--|#', '', s)
        elif t == 'removeNulls': s = s.replace('\x00', '')
        elif t == 'removeWhitespace': s = re.sub(r'\s', '', s)
        elif t == 'lowercase': s = s.lower()
        # 'none' dan 'utf8toUnicode' tidak mengubah str Python
    return s

def load_crs(conf_path=None):
    """Mengurai REQUEST-942 dari OWASP CRS. Yang dipakai: aturan @rx dan @detectSQLi yang menyasar ARGS
    (nilai parameter), tanpa 'chain'. Aturan berantai dan aturan khusus header/cookie/filename dicatat
    sebagai dikecualikan."""
    import regex
    txt = open(conf_path).read() if conf_path else urllib.request.urlopen(CRS_URL, timeout=60).read().decode()
    txt = re.sub(r'\\\n\s*', '', txt)
    rules, skipped = [], []
    for line in txt.split('\n'):
        if not line.startswith('SecRule'): continue
        m = re.match(r'SecRule\s+(\S+)\s+"((?:[^"\\]|\\.)*)"\s+"(.*)"\s*$', line)
        if not m: continue
        var, op, act = m.groups()
        rid = re.search(r'id:(\d+)', act); pl = re.search(r"paranoia-level/(\d)", act)
        if not (rid and pl): continue                                   # baris kontrol PL (@lt)
        rid, pl = rid.group(1), int(pl.group(1))
        sev = re.search(r"severity:'(\w+)'", act); sev = sev.group(1) if sev else 'CRITICAL'
        score = {'CRITICAL': 5, 'ERROR': 4, 'WARNING': 3, 'NOTICE': 2}.get(sev, 5)
        targets = var.split('|')
        if 'chain' in act.split(','):  skipped.append((rid, pl, 'chain')); continue
        if 'ARGS' not in targets:      skipped.append((rid, pl, 'bukan ARGS')); continue
        tfs = re.findall(r't:(\w+)', act)
        if op.startswith('@detectSQLi'):
            rules.append(dict(id=rid, pl=pl, score=score, kind='libinjection', tfs=tfs)); continue
        if not op.startswith('@rx'):   skipped.append((rid, pl, op.split()[0])); continue
        pat = op[4:].replace('\\"', '"')
        pat = pat.replace(r'\x{c2}\x{b4}', '´').replace(r'\x{e2}\x80[\x98\x99]', '[‘’]')
        rules.append(dict(id=rid, pl=pl, score=score, kind='rx', tfs=tfs, rx=regex.compile(pat)))
    return rules, skipped, ('local:' + conf_path) if conf_path else CRS_URL

def crs_detector(rules, pl, libinj):
    """True bila skor anomali (jumlah skor aturan yang cocok, PL <= pl) >= 5 (ambang inbound bawaan CRS)."""
    act = [r for r in rules if r['pl'] <= pl and (r['kind'] == 'rx' or libinj is not None)]
    stats = {'timeouts': 0}
    def f(s):
        total = 0
        for r in act:
            v = _crs_transform(s, r['tfs'])
            if r['kind'] == 'libinjection': hit = libinj(v)
            else:
                try: hit = r['rx'].search(v, timeout=1.0) is not None
                except TimeoutError: hit = False; stats['timeouts'] += 1
            if hit:
                total += r['score']
                if total >= 5: return True
        return False
    return f, len(act), stats

# =====================================================================================
# 5. Metrik
# =====================================================================================
def kaggle_metrics(y, p):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return dict(Accuracy=100 * accuracy_score(y, p), Precision=100 * precision_score(y, p, zero_division=0),
                Recall=100 * recall_score(y, p), F1=100 * f1_score(y, p), FPR=100 * fp / (fp + tn), FP=int(fp), FN=int(fn))

def mcnemar(a, b, y=None):
    """Uji McNemar eksak (binomial) untuk dua prediksi berpasangan pada baris yang sama.
    Bila y diberikan: benar/salah terhadap y; bila tidak (sqlmap, semua serangan): terdeteksi/tidak."""
    ca, cb = (a == y, b == y) if y is not None else (a == 1, b == 1)
    n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    p = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    return n01, n10, p

# =====================================================================================
# 6. Main
# =====================================================================================
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', required=True); ap.add_argument('--sqlmap', required=True)
    ap.add_argument('--hp', default=None, help='best_hyperparameters.csv dari run REV9 (dianjurkan)')
    ap.add_argument('--ref', default=None, help='folder hasil REV9 untuk cek reproduksi angka')
    ap.add_argument('--crs', default=None, help='file REQUEST-942 lokal (opsional; default unduh tag ' + CRS_TAG + ')')
    ap.add_argument('--models', default=','.join(MAIN + LINEAR))
    ap.add_argument('--out', default='results_rev10')
    a = ap.parse_args(); os.makedirs(a.out, exist_ok=True)
    names = [m.strip() for m in a.models.split(',')]
    info = dict(python=platform.python_version(), sklearn=sklearn.__version__, xgboost=xgboost.__version__,
                numpy=np.__version__, pandas=pd.__version__, seed=RS, crs_tag=CRS_TAG)

    # ---- data dan split (identik dengan REV9) ----
    df, dmeta = load_kaggle(a.data); info.update(data=dmeta)
    X, y = df.norm.values, df.label.values
    tr, te = train_test_split(np.arange(len(y)), test_size=TEST_SIZE, stratify=y, random_state=RS)
    assert not (set(X[tr]) & set(X[te]))
    S, smeta = load_sqlmap(a.sqlmap, X); info.update(sqlmap=smeta)
    log('Kaggle', dmeta, '| train', len(tr), 'test', len(te), '| sqlmap', smeta)

    # ---- kondisi B: normalisasi operator pada baris yang sama ----
    Xo = np.array([normalize_sql_operators(t) for t in X]); So = S.norm.map(normalize_sql_operators).values
    tr_set = set(Xo[tr])
    coll_te = np.array([t in tr_set for t in Xo[te]])                    # baris uji yang identik dg baris latih setelah normalisasi
    g = pd.DataFrame({'t': Xo, 'y': y}).groupby('t').y.nunique()
    all_o = set(Xo); coll_sq = np.array([t in all_o for t in So])
    leak = dict(test_rows_identical_to_train_after_norm=int(coll_te.sum()),
                label_conflicting_texts_after_norm=int((g > 1).sum()),
                sqlmap_strings_identical_to_kaggle_after_norm=int(coll_sq.sum()),
                sqlmap_with_spaced_equals_before=float(np.mean([' = ' in t for t in S.norm]) * 100),
                sqlmap_with_spaced_equals_after=float(np.mean([' = ' in t for t in So]) * 100),
                kaggle_test_sqli_spaced_equals_before=float(np.mean([' = ' in t for t in X[te][y[te] == 1]]) * 100),
                kaggle_test_sqli_spaced_equals_after=float(np.mean([' = ' in t for t in Xo[te][y[te] == 1]]) * 100))
    info['normalization_check'] = leak; log('normalisasi', leak)

    # ---- hyperparameter ----
    HP = load_hp(a.hp); hp_rows = []
    for n in names:
        if n in HP: src = 'REV9 (' + os.path.basename(a.hp) + ')'; cvs = np.nan
        else:
            if MODELS[n]['space'] is None: sys.exit(f'Hyperparameter {n} tidak ada di --hp; jalankan dengan --hp dari REV9.')
            HP[n], cvs = tune(n, X, y, tr); src = 'dituning ulang (protokol REV9)'
        hp_rows.append(dict(Model=n, params=json.dumps({k[5:]: v for k, v in HP[n].items()}), source=src, cv_macro_f1=cvs))
        log('HP', hp_rows[-1])
    pd.DataFrame(hp_rows).to_csv(f'{a.out}/hyperparameters_rev10.csv', index=False)

    # ---- latih dan prediksi: kondisi A (tanpa) dan B (dengan normalisasi operator) ----
    P = {}; rows = []
    for cond, XX, SS in [('Tanpa normalisasi operator', X, S.norm.values), ('Dengan normalisasi operator', Xo, So)]:
        for n in names:
            p = make_pipe(n, HP[n]); t0 = time.time(); p.fit(XX[tr], y[tr], **fit_kw(n, y[tr])); tt = time.time() - t0
            pk = p.predict(XX[te]); ps = p.predict(SS); P[(cond, n)] = (pk, ps)
            rows.append(dict(Condition=cond, Model=n, Type='ML (TF-IDF char 1-3)', **kaggle_metrics(y[te], pk),
                             sqlmap_DR=100 * ps.mean(), train_s=tt))
            log(cond, '|', n, '|', ', '.join(f'{k}={float(v):.3f}' if isinstance(v, (float, np.floating)) else f'{k}={v}'
                                       for k, v in rows[-1].items() if k not in ('Condition', 'Model', 'Type')))

    # ---- baseline rule-based (dievaluasi pada kedua bentuk input juga) ----
    libinj, libver = libinjection_fn(); info['libinjection'] = libver
    rules, skipped, crs_src = load_crs(a.crs)
    info['crs'] = dict(source=crs_src, rules_used=[(r['id'], r['pl'], r['kind']) for r in rules], rules_skipped=skipped)
    RB = {}
    if libinj: RB['libinjection'] = (libinj, 'Signature (tokenizer)')
    for pl in (1, 2):
        f, nr, st = crs_detector(rules, pl, libinj); RB[f'OWASP CRS {CRS_TAG} PL{pl}'] = (f, f'Rule-based ({nr} aturan, ambang 5)')
        info['crs'][f'PL{pl}_n_rules'] = nr; info['crs'][f'PL{pl}_regex_stats'] = st
    for cond, XX, SS in [('Tanpa normalisasi operator', X, S.norm.values), ('Dengan normalisasi operator', Xo, So)]:
        for n, (f, typ) in RB.items():
            t0 = time.time(); pk = np.array([int(f(s)) for s in XX[te]]); ps = np.array([int(f(s)) for s in SS])
            P[(cond, n)] = (pk, ps)
            rows.append(dict(Condition=cond, Model=n, Type=typ, **kaggle_metrics(y[te], pk), sqlmap_DR=100 * ps.mean(), train_s=np.nan))
            log(cond, '|', n, f"| F1={rows[-1]['F1']:.2f} FPR={rows[-1]['FPR']:.2f} sqlmap={rows[-1]['sqlmap_DR']:.2f} ({time.time() - t0:.0f} s)")
    R = pd.DataFrame(rows); R.to_csv(f'{a.out}/all_results_rev10.csv', index=False)

    # ---- alarm palsu pada input benign yang berupa pernyataan SQL lengkap vs input biasa ----
    STMT = re.compile(r'^\s*(select|insert|update|delete|create|drop|alter|with)\b')
    ben = y[te] == 0; is_stmt = np.array([bool(STMT.search(s)) for s in X[te]])
    fpb = []
    for n in list(names) + list(RB):
        pk = P[('Tanpa normalisasi operator', n)][0]; fpm = ben & (pk == 1)
        fpb.append(dict(Model=n, benign_test=int(ben.sum()), benign_sql_statements=int((ben & is_stmt).sum()),
                        FP=int(fpm.sum()), FP_on_sql_statements=int((fpm & is_stmt).sum()),
                        FPR_sql_statements=100 * fpm[ben & is_stmt].mean(), FPR_other_benign=100 * fpm[ben & ~is_stmt].mean()))
    FB = pd.DataFrame(fpb); FB.to_csv(f'{a.out}/false_alarms_by_input_type.csv', index=False)
    log('\nAlarm palsu menurut jenis input benign (tanpa normalisasi operator):\n' + FB.round(2).to_markdown(index=False))

    # ---- uji berpasangan sebelum vs sesudah + evaluasi tanpa baris yang identik dg data latih ----
    pair = []
    for n in list(names) + list(RB):
        (ka, sa), (kb, sb) = P[('Tanpa normalisasi operator', n)], P[('Dengan normalisasi operator', n)]
        k01, k10, kp = mcnemar(ka, kb, y[te]); s01, s10, sp = mcnemar(sa, sb)
        keep = ~coll_te; keep_s = ~coll_sq
        pair.append(dict(Model=n, kaggle_only_before_correct=k01, kaggle_only_after_correct=k10, kaggle_mcnemar_p=kp,
                         sqlmap_only_before_detected=s01, sqlmap_only_after_detected=s10, sqlmap_mcnemar_p=sp,
                         F1_after_excl_overlap=100 * f1_score(y[te][keep], kb[keep]),
                         FPR_after_excl_overlap=kaggle_metrics(y[te][keep], kb[keep])['FPR'],
                         sqlmap_DR_after_excl_overlap=100 * sb[keep_s].mean()))
    PT = pd.DataFrame(pair); PT.to_csv(f'{a.out}/paired_tests_rev10.csv', index=False)

    # ---- cek reproduksi terhadap angka REV9 ----
    if a.ref and os.path.exists(f'{a.ref}/results_all_splits.csv'):
        ref = pd.read_csv(f'{a.ref}/results_all_splits.csv'); ref = ref[ref.Split == '80:20'].set_index('Algorithm')
        ext = json.load(open(f'{a.ref}/external_sqlmap_summary.json'))['detection_all']
        chk = []
        for n in [m for m in MAIN if m in names]:
            r = R[(R.Condition == 'Tanpa normalisasi operator') & (R.Model == n)].iloc[0]
            chk.append(dict(Model=n, F1_REV9=100 * ref.loc[n, 'F1'], F1_now=r.F1, sqlmap_REV9=ext[n], sqlmap_now=r.sqlmap_DR))
        C = pd.DataFrame(chk); C['match'] = (abs(C.F1_REV9 - C.F1_now) < 0.005) & (abs(C.sqlmap_REV9 - C.sqlmap_now) < 0.005)
        C.to_csv(f'{a.out}/reproduction_check.csv', index=False); info['reproduces_rev9'] = bool(C.match.all())
        log('\nCek reproduksi REV9:\n' + C.round(2).to_markdown(index=False))

    # ---- Tabel A dan B ----
    A = R[R.Condition == 'Tanpa normalisasi operator'].copy()
    A['Kelompok'] = A.Model.map(lambda m: 'Model utama' if m in MAIN else 'Baseline')
    TA = A[['Kelompok', 'Model', 'Type', 'Accuracy', 'Precision', 'Recall', 'F1', 'FPR', 'FP', 'FN', 'sqlmap_DR']].rename(
        columns={'Type': 'Jenis', 'Accuracy': 'Akurasi (%)', 'Precision': 'Presisi (%)', 'Recall': 'Recall (%)', 'F1': 'F1 (%)',
                 'FPR': 'FPR (%)', 'sqlmap_DR': 'Deteksi sqlmap (%)'})
    b0 = R[R.Condition == 'Tanpa normalisasi operator'].set_index('Model'); b1 = R[R.Condition == 'Dengan normalisasi operator'].set_index('Model')
    order = [m for m in list(names) + list(RB)]
    TB = pd.DataFrame({'Model': order,
        'F1 sebelum (%)': b0.loc[order, 'F1'].values, 'F1 sesudah (%)': b1.loc[order, 'F1'].values,
        'FPR sebelum (%)': b0.loc[order, 'FPR'].values, 'FPR sesudah (%)': b1.loc[order, 'FPR'].values,
        'p McNemar (Kaggle)': PT.set_index('Model').loc[order, 'kaggle_mcnemar_p'].values,
        'sqlmap sebelum (%)': b0.loc[order, 'sqlmap_DR'].values, 'sqlmap sesudah (%)': b1.loc[order, 'sqlmap_DR'].values,
        'p McNemar (sqlmap)': PT.set_index('Model').loc[order, 'sqlmap_mcnemar_p'].values})
    TB['Δ sqlmap (pp)'] = TB['sqlmap sesudah (%)'] - TB['sqlmap sebelum (%)']
    TA.to_csv(f'{a.out}/table_A_baselines.csv', index=False); TB.to_csv(f'{a.out}/table_B_operator_norm.csv', index=False)

    def fmt_p(v): return '< 0.001' if v < 0.001 else f'{v:.3f}'
    TBp = TB.copy()
    for c in ['p McNemar (Kaggle)', 'p McNemar (sqlmap)']: TBp[c] = TBp[c].map(fmt_p)
    md = ['## Tabel A. Model utama vs baseline (tanpa normalisasi operator); test set Kaggle 80:20 (n = %d) dan %d string sqlmap\n' % (len(te), len(S)),
          TA.to_markdown(index=False, floatfmt='.2f'),
          '\n\n## Tabel B. Sebelum vs sesudah normalize_sql_operators() pada data latih, test set Kaggle, dan payload sqlmap\n',
          TBp.to_markdown(index=False, floatfmt='.2f'),
          '\n\nCatatan B: hyperparameter sama dengan kondisi tanpa normalisasi; split baris identik. Setelah normalisasi, '
          f"{leak['test_rows_identical_to_train_after_norm']} baris uji identik dengan baris latih dan "
          f"{leak['sqlmap_strings_identical_to_kaggle_after_norm']} string sqlmap identik dengan teks Kaggle "
          '(lihat paired_tests_rev10.csv untuk angka tanpa baris tersebut).']
    open(f'{a.out}/tables_rev10.md', 'w').write('\n'.join(md))
    json.dump(info, open(f'{a.out}/run_info_rev10.json', 'w'), indent=2, default=str)
    print('\n' + '\n'.join(md))
    print('\nUji berpasangan & angka tanpa overlap:\n' + PT.round(4).to_markdown(index=False))

if __name__ == '__main__':
    main()


In [ ]:
!python /content/sqli_baselines_norm.py --data "$DATA_PATH" --sqlmap /content/sqlmap_payloads.csv \
    --hp /content/ref_rev9/best_hyperparameters.csv --ref /content/ref_rev9 --out /content/results_rev10

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/results_rev10', 'zip', '/content/results_rev10')
files.download('/content/results_rev10.zip')